In [1]:
import os
import re
import glob
import pandas as pd

BASE_DIR = "."  # folder with the CSVs
OUT_FILE = os.path.join(BASE_DIR, "all informed llm responses.csv")

# grab all "* informed responses.csv"
paths = sorted(glob.glob(os.path.join(BASE_DIR, "* informed responses.csv")))
if not paths:
    raise SystemExit("No '* informed responses.csv' files found.")

def llm_name_from_filename(path: str) -> str:
    # "gpt52 informed responses.csv" -> "gpt52"
    base = os.path.basename(path)
    name = re.sub(r"\s*informed\s+responses\.csv$", "", base, flags=re.IGNORECASE)
    return name.strip()

frames = []
for p in paths:
    llm = llm_name_from_filename(p)
    df = pd.read_csv(p)

    if "number" not in df.columns or "choice" not in df.columns:
        raise ValueError(f"{p} must contain columns 'number' and 'choice'")

    tmp = df[["number", "choice"]].copy()
    tmp["number"] = pd.to_numeric(tmp["number"], errors="coerce").astype("Int64")
    tmp = tmp.dropna(subset=["number"])
    tmp = tmp.drop_duplicates(subset=["number"], keep="last")
    tmp = tmp.rename(columns={"choice": llm})

    frames.append(tmp)

# outer join on question number
merged = frames[0]
for f in frames[1:]:
    merged = merged.merge(f, on="number", how="outer")

# number first, then model columns sorted
model_cols = sorted([c for c in merged.columns if c != "number"])
merged = merged[["number"] + model_cols].sort_values("number")

merged.to_csv(OUT_FILE, index=False)
print("Wrote:", OUT_FILE)

merged.head()


Wrote: .\all informed llm responses.csv


,number,claude35sonnet,claudesonnet45,deepseekv32,gpt41,gpt52,gpt5pro,llama323binstruct,qwen3maxthinking
0,1,D,D,D,D,D,D,D,D
1,2,D,D,D,D,D,D,D,D
2,3,D,D,D,D,D,D,D,D
3,4,B,B,B,B,NaN,D,D,B
4,5,D,D,D,D,D,D,D,D


In [2]:
import os
import re
import glob
import pandas as pd

BASE_DIR = "."
PATTERN = os.path.join(BASE_DIR, "* responses.csv")  # includes both normal + informed

paths = sorted(glob.glob(PATTERN))
if not paths:
    raise SystemExit("No '* responses.csv' files found.")

def model_name_from_filename(path: str) -> str:
    # "gpt52 informed responses.csv" -> "gpt52 (informed)"
    # "gpt52 responses.csv"          -> "gpt52"
    base = os.path.basename(path)
    m = re.match(r"(.+?)\s+(informed\s+)?responses\.csv$", base, flags=re.IGNORECASE)
    if not m:
        return os.path.splitext(base)[0]
    name = m.group(1).strip()
    informed = bool(m.group(2))
    return f"{name} (informed)" if informed else name

counts_by_model = {}

for p in paths:
    df = pd.read_csv(p)
    if "choice" not in df.columns:
        continue  # skip anything unexpected

    s = df["choice"].astype(str).str.upper().str.strip()
    vc = s.value_counts()

    counts_by_model[model_name_from_filename(p)] = {
        "A": int(vc.get("A", 0)),
        "B": int(vc.get("B", 0)),
        "C": int(vc.get("C", 0)),
        "D": int(vc.get("D", 0)),
        "E": int(vc.get("E", 0)),
    }

table = pd.DataFrame.from_dict(counts_by_model, orient="index")[["A","B","C","D","E"]]
table["TOTAL"] = table.sum(axis=1)

# Sort by name (or by D count if you want)
table = table.sort_index()

# Save
out_path = os.path.join(BASE_DIR, "option_counts_all_models.csv")
table.to_csv(out_path)

print("Wrote:", out_path)
table


Wrote: .\option_counts_all_models.csv


,A,B,C,D,E,TOTAL
claude35sonnet (informed),5,69,1,62,13,150
claudesonnet45 (informed),0,38,4,107,1,150
deepseekv32 (informed),0,3,0,147,0,150
gpt41 (informed),0,2,1,147,0,150
gpt52 (informed),0,9,1,138,0,148
gpt5pro (informed),0,16,2,132,0,150
llama323binstruct (informed),0,1,0,149,0,150
qwen3maxthinking (informed),0,27,3,120,0,150


In [3]:
import pandas as pd

# Informed merged choices file (one column per model, one row per question)
INPUT = "all informed llm responses.csv"   # or r"/mnt/data/all informed llm responses.csv"
OUT   = "informed_counts_by_hierarchy.csv"

df = pd.read_csv(INPUT)

model_cols = [c for c in df.columns if c != "number"]

# hierarchy split: first 75 = lower, last 75 = higher
df["hierarchy"] = df["number"].apply(lambda n: "Lower (1–75)" if int(n) <= 75 else "Higher (76–150)")

# clean choices
for c in model_cols:
    df[c] = df[c].astype(str).str.upper().str.strip()
    df.loc[~df[c].isin(["A", "B", "C", "D", "E"]), c] = pd.NA

OPTIONS = ["A", "B", "C", "D", "E"]

def counts_table(sub_df: pd.DataFrame) -> pd.DataFrame:
    ct = sub_df[model_cols].apply(lambda s: s.value_counts(dropna=True))
    ct = ct.reindex(OPTIONS).fillna(0).astype(int)
    return ct

counts_lower  = counts_table(df[df["hierarchy"] == "Lower (1–75)"])
counts_higher = counts_table(df[df["hierarchy"] == "Higher (76–150)"])

combined = pd.concat(
    {"Lower (1–75)": counts_lower, "Higher (76–150)": counts_higher},
    axis=0
)
combined.index.names = ["Group", "Option"]

# optional difference block: Higher - Lower
diff = (counts_higher - counts_lower).reindex(OPTIONS)
combined = pd.concat([combined, pd.concat({"Δ (Higher−Lower)": diff}, axis=0)])
combined.index.names = ["Group", "Option"]

combined.to_csv(OUT)
print("Wrote:", OUT)

combined


Wrote: informed_counts_by_hierarchy.csv


claude35sonnet  claudesonnet45  deepseekv32  gpt41  \
Group            Option                                                       
Lower (1–75)     A                    0               0            0      0   
                 B                   30              14            3      2   
                 C                    1               3            0      1   
                 D                   44              58           72     72   
                 E                    0               0            0      0   
Higher (76–150)  A                    5               0            0      0   
                 B                   39              24            0      0   
                 C                    0               1            0      0   
                 D                   18              49           75     75   
                 E                   13               1            0      0   
Δ (Higher−Lower) A                    5               0            0      0   
                 B                    9              10           -3     -2   
                 C                   -1              -2            0     -1   
                 D                  -26              -9            3      3   
                 E                   13               1            0      0   

                         gpt52  gpt5pro  llama323binstruct  qwen3maxthinking  
Group            Option                                                       
Lower (1–75)     A           0        0                  0                 0  
                 B           1        5                  1                 9  
                 C           1        1                  0                 3  
                 D          71       69                 74                63  
                 E           0        0                  0                 0  
Higher (76–150)  A           0        0                  0                 0  
                 B           8       11                  0                18  
                 C           0        1                  0                 0  
                 D          67       63                 75                57  
                 E           0        0                  0                 0  
Δ (Higher−Lower) A           0        0                  0                 0  
                 B           7        6                 -1                 9  
                 C          -1        0                  0                -3  
                 D          -4       -6                  1                -6  
                 E           0        0                  0                 0